# Optuna `optuna_nogate` — TCN/LSTM sem persistence gate (Kaggle)

Continua o estudo de hiperparâmetros iniciado localmente, sem persistence gate
e com `context_mode: horizon` no TCN/TCN_BI.

**Setup no Kaggle:**
1. Empacote o projeto localmente: `bash scripts/kaggle_package.sh`
2. Suba `dist/kaggle_optuna_<data>.zip` como **Dataset** (ex.: `wind-speed-forecasting`)
3. *(Opcional, para retomar)* Suba o `optuna_nogate.zip` de uma execução anterior
   como outro Dataset (ex.: `optuna-progress`)
4. Anexe os dois Datasets a este notebook e selecione **Accelerator: GPU T4 x2**
   (ou P100) e **Internet: ON** (para o pip)
5. **Save & Run All (Commit)** — sessões GPU duram até 12 h; o estudo grava cada
   trial no sqlite continuamente, então se cortar no limite basta baixar o
   `optuna_nogate.zip` da versão salva, subir como Dataset e rodar de novo

Quota Kaggle: ~30 h de GPU/semana. O restante do estudo (~10 h) cabe em uma
sessão. A GPU T4 tem 16 GB — menos risco de OOM que os 6 GB locais, e o
`step_optuna` já poda trials que estourarem memória.

In [ ]:
!nvidia-smi
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
%pip install -q "tensorflow[and-cuda]==2.21.0" "keras==3.12.1" "keras-tcn==3.5.6" "optuna==4.9.0" "PyWavelets==1.8.0" "pandas>=2.0" "scikit-learn>=1.3" "numpy>=1.26"
import tensorflow as tf
print("TF apos pin:", tf.__version__)

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path("/kaggle/working")
REPO = WORK / "wind-speed-forecasting"
REPO.mkdir(parents=True, exist_ok=True)

print("Conteudo disponivel em /kaggle/input:")
for p in sorted(INPUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(INPUT))

zips = sorted(INPUT.rglob("*.zip"))
repo_zip = next((z for z in zips if "kaggle_optuna_" in z.name), None)

if repo_zip is not None:
    print("\nrepo zip:", repo_zip)
    with zipfile.ZipFile(repo_zip) as z:
        z.extractall(REPO)
else:
    # O Kaggle pode auto-extrair o upload: procurar a raiz do repo extraido
    # (o ancestral de data/dataset.csv que contem src/ ou pipeline/).
    roots = set()
    for csv in INPUT.rglob("dataset.csv"):
        d = csv.parent
        while d != d.parent:
            if (d / "src").is_dir() or (d / "pipeline").is_dir():
                roots.add(d)
                break
            d = d.parent
    roots = sorted(roots)
    if not roots:
        raise FileNotFoundError(
            "Nem kaggle_optuna_*.zip nem um repo extraido (data/dataset.csv)"
            " foram encontrados em /kaggle/input. Faca upload do pacote gerado"
            " por `bash scripts/kaggle_package.sh` como Dataset (Datasets ->"
            " New Dataset) e anexe-o em Add Input -> Your Datasets."
        )
    src_root = roots[0]
    print("\nrepo ja extraido pelo Kaggle em:", src_root, "-> copiando")
    shutil.copytree(
        src_root, REPO, dirs_exist_ok=True,
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".venv", ".git"),
    )

def _restore_progress():
    progress_zip = next(
        (z for z in zips if "optuna_nogate" in z.name and "kaggle_optuna_" not in z.name),
        None,
    )
    if progress_zip is not None:
        print("retomando progresso (zip):", progress_zip)
        with zipfile.ZipFile(progress_zip) as z:
            z.extractall(REPO)
        return True
    target = REPO / "pipeline" / "tmp" / "optuna_nogate"
    db_dirs = {p.parent for p in INPUT.rglob("optuna.db")}
    if db_dirs:
        target.mkdir(parents=True, exist_ok=True)
        for d in db_dirs:
            print("retomando progresso (pasta):", d)
            shutil.copytree(d, target / d.name, dirs_exist_ok=True)
        return True
    return False

if _restore_progress():
    print("progresso anterior restaurado")
else:
    print("\nsem progresso separado — usando o que vier dentro do pacote")

assert (REPO / "data" / "dataset.csv").is_file(), "dataset.csv ausente no pacote"
dbs = sorted(REPO.glob("pipeline/tmp/optuna_nogate/*/optuna.db"))
print("repo em:", REPO)
print("dbs de estudo:", [str(d.relative_to(REPO)) for d in dbs] or "nenhum (do zero)")


In [ ]:
import os
import subprocess
import sys
import time

os.chdir("/kaggle/working/wind-speed-forecasting")
ngpus = len([l for l in os.popen("nvidia-smi -L").read().splitlines() if "GPU" in l])
print("GPUs detectadas:", ngpus)

if ngpus >= 2:
    jobs = [(0, ["tcn"]), (1, ["tcn_bi"])]
else:
    jobs = [(None, ["tcn", "tcn_bi"])]

procs = []
for gpu, wrappers in jobs:
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log_path = f"/kaggle/working/log_{'_'.join(wrappers)}.txt"
    log = open(log_path, "w")
    cmd = [sys.executable, "pipeline/step_optuna.py",
           "--config", "pipeline/pipeline.optuna_nogate.json",
           "--wrappers", *wrappers]
    procs.append(subprocess.Popen(cmd, env=env, stdout=log,
                                  stderr=subprocess.STDOUT))
    print(f"GPU {gpu}: {' '.join(wrappers)} -> PID {procs[-1].pid} | log {log_path}")

codes = [p.wait() for p in procs]
print("codigos de saida:", codes)

import glob
for log in glob.glob("/kaggle/working/log_*.txt"):
    print("\n=== ultimas linhas de", log, "===")
    print("".join(open(log).readlines()[-6:]))


In [ ]:
import glob
import json

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
for db in sorted(glob.glob("pipeline/tmp/optuna_nogate/*/optuna.db")):
    name = db.split("/")[-2]
    study = optuna.load_study(study_name="optuna_nogate", storage=f"sqlite:///{db}")
    done = [t for t in study.trials if t.state.name == "COMPLETE"]
    best = min((t.value for t in done), default=float("nan"))
    print(f"{name}: {len(study.trials)}/100 trials | melhor block_mse={best:.4f}")

for f in sorted(glob.glob("pipeline/tmp/optuna_nogate/*/best_trial.json")):
    with open(f) as handle:
        bt = json.load(handle)
    print(f"\n{f}:\n{json.dumps(bt['best_params'], indent=2)}")

In [ ]:
!cd /kaggle/working/wind-speed-forecasting && \
    zip -qr /kaggle/working/optuna_nogate.zip pipeline/tmp/optuna_nogate -x "*__pycache__*" && \
    ls -lh /kaggle/working/optuna_nogate.zip
print("Baixe o zip da versao salva para retomar o estudo depois (ou treinar local).")